In [2]:
import datetime
from datetime import timedelta
import sys

import sqlite3
import numpy as np
import matplotlib.pyplot as plt

In [12]:
weekStarts   = []
weekEnds     = []
sbWeekStarts = []
sbWeekEnds   = []

days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

seasonStart = datetime.datetime(2025, 3, 18)
seasonEnd   = datetime.datetime(2025, 9, 21)

seasonStartDayNumber = seasonStart.weekday()

print(seasonStartDayNumber)
print(days[seasonStartDayNumber])

daysToAdd = 0

daysToAdd = timedelta(days = 6 - seasonStartDayNumber)

endFirstWeek = seasonStart + daysToAdd

endFirstWeekNumber = endFirstWeek.weekday()

#print(endFirstWeek)
#print(endFirstWeekNumber)
#print(days[endFirstWeekNumber])

formattedCurWeekStart = seasonStart   .strftime("%Y-%m-%d")
formattedCurWeekEnd   = endFirstWeek  .strftime("%Y-%m-%d")

formattedSbCurWeekStart = seasonStart   .strftime("%Y%m%d")
formattedSbCurWeekEnd   = endFirstWeek  .strftime("%Y%m%d")

weekStarts.append(formattedCurWeekStart )
weekEnds  .append(formattedCurWeekEnd   )

sbWeekStarts.append(formattedSbCurWeekStart)
sbWeekEnds  .append(formattedSbCurWeekEnd  )

print(f"week 1: {formattedCurWeekStart} -> {formattedCurWeekEnd}")

curWeekStart = endFirstWeek + timedelta(days=1)
curWeekEnd   = curWeekStart + timedelta(days=6)

#print(curWeekStart)
#print(curWeekEnd)

i = 2

while curWeekStart <= seasonEnd:
    formattedCurWeekStart = curWeekStart.strftime("%Y-%m-%d")
    formattedCurWeekEnd   = curWeekEnd  .strftime("%Y-%m-%d")

    formattedSbCurWeekStart = curWeekStart.strftime("%Y%m%d")
    formattedSbCurWeekEnd   = curWeekEnd  .strftime("%Y%m%d")
    
    print(f"week {i}: {formattedCurWeekStart} -> {formattedCurWeekEnd}")

    weekStarts.append(formattedCurWeekStart)
    weekEnds  .append(formattedCurWeekEnd  )

    sbWeekStarts.append(formattedSbCurWeekStart)
    sbWeekEnds  .append(formattedSbCurWeekEnd  )
    
    curWeekStart = curWeekEnd   + timedelta(days=1)
    curWeekEnd   = curWeekStart + timedelta(days=6)

    if curWeekEnd > seasonEnd:
        curWeekEnd = seasonEnd

    i = i + 1

1
Tuesday
week 1: 2025-03-18 -> 2025-03-23
week 2: 2025-03-24 -> 2025-03-30
week 3: 2025-03-31 -> 2025-04-06
week 4: 2025-04-07 -> 2025-04-13
week 5: 2025-04-14 -> 2025-04-20
week 6: 2025-04-21 -> 2025-04-27
week 7: 2025-04-28 -> 2025-05-04
week 8: 2025-05-05 -> 2025-05-11
week 9: 2025-05-12 -> 2025-05-18
week 10: 2025-05-19 -> 2025-05-25
week 11: 2025-05-26 -> 2025-06-01
week 12: 2025-06-02 -> 2025-06-08
week 13: 2025-06-09 -> 2025-06-15
week 14: 2025-06-16 -> 2025-06-22
week 15: 2025-06-23 -> 2025-06-29
week 16: 2025-06-30 -> 2025-07-06
week 17: 2025-07-07 -> 2025-07-13
week 18: 2025-07-14 -> 2025-07-20
week 19: 2025-07-21 -> 2025-07-27
week 20: 2025-07-28 -> 2025-08-03
week 21: 2025-08-04 -> 2025-08-10
week 22: 2025-08-11 -> 2025-08-17
week 23: 2025-08-18 -> 2025-08-24
week 24: 2025-08-25 -> 2025-08-31
week 25: 2025-09-01 -> 2025-09-07
week 26: 2025-09-08 -> 2025-09-14
week 27: 2025-09-15 -> 2025-09-21


In [4]:
def get_percentile(sortedList, value):
    arr = np.array(sortedList)
    count_less_than = np.sum(arr < value)
    percentile = (count_less_than / len(arr)) * 100
    return percentile

In [5]:
def get_reverse_percentile(sortedList, value):
    arr = np.array(sortedList)
    count_more_than = np.sum(arr > value)
    percentile = (count_more_than / len(arr)) * 100
    return percentile

In [6]:
def normalize(num, maxValue, minValue):
    if maxValue == 0 and minValue == 0:
        return 0
    if maxValue == minValue:
        return 0
    
    normalizedValue = (num - minValue) / (maxValue - minValue)

    return normalizedValue

In [7]:
def reverse_normalization(num, maxValue, minValue):
    if maxValue == 0 and minValue == 0:
        return 0
    if maxValue == minValue:
        return 0
    
    reverseNormalizedValue = 1 - ((num - minValue) / (maxValue - minValue))

    return reverseNormalizedValue

In [34]:
teamToGamesPlayed = {}

homeGameQuery = "SELECT hometeam as team,COUNT(gid) as homeGames from games where date between \"2025-03-18\" AND \"2025-03-23\" group by games.hometeam"
awayGameQuery = "SELECT visteam as team,COUNT(gid) as awayGames from games where date between \"2025-03-18\" AND \"2025-03-23\" group by games.visteam"

connection = sqlite3.connect('player_batting.db')
cursor     = connection.cursor()

cursor.execute(homeGameQuery)

rows = cursor.fetchall()

for row in rows:
    team = row[0]
    games = int(row[1])

    teamToGamesPlayed[team] = games

cursor.execute(awayGameQuery)

rows = cursor.fetchall()

for row in rows:
    team = row[0]
    games = int(row[1])

    if team in teamToGamesPlayed:
        teamToGamesPlayed[team] = teamToGamesPlayed[team] + games
    else:
        teamToGamesPlayed[team] = games

cursor.close()
connection.close()


In [35]:
for key in teamToGamesPlayed:
    print(f"{key}: {teamToGamesPlayed[key]}")

CHN: 2
LAN: 2


In [15]:
#ADD ROW FOR PERCENTILE AGAINST ALL PLAYERS AND ONE AGAINST ALL QUALIFIED PLAYERS UP TO THAT POINT IN THE SEASON
#NEED TO DETERMINE IF A PLAYER IS A ROOKIE OR NOT (CAN MAYBE USE MLB API FOR THIS)
#NEED 1.0 IP PER GAME FOR A STARTER, 0.5 IP PER GAME FOR A RELIEVER OR ROOKIE

eraQuery = "SELECT id,sum(p_er),sum(p_ipouts) FROM pitching WHERE gametype = \"regular\" AND p_gs=\"1\" AND date BETWEEN \"2025-03-18\" AND\""

addAggregateGradeStatement = "INSERT INTO aggregate_weekly_grades (week_number,week_start,week_end,player_id,stat,num,percentile,season) VALUES("

playerToAggregateEra        = {}
playerToAggregatePercentile = {}

for i in range(len(weekStarts)):
    aggregatePlayerToEra = {}

    weekStart = weekStarts[i]
    weekEnd   = weekEnds  [i]

    playerToAggregateEra       [i+1] = {}
    playerToAggregatePercentile[i+1] = {}
    
    playerEraQuery = eraQuery + weekEnd + "\" GROUP BY id"

    connection = sqlite3.connect('player_batting.db')
    cursor     = connection.cursor()

    cursor.execute(playerEraQuery)

    rows = cursor.fetchall()
    
    for row in rows:
        playerId    = row[0]
        earnedRuns  = float(row[1])
        outs        = float(row[2])

        era = 0.0

        if outs == 0.0 and earnedRuns > 0.0:
            era = sys.float_info.max
        elif outs == 0.0 and earnedRuns == 0.0:
            era = 0.0
        else:
            era = (earnedRuns * 9.0) / (outs / 3.0)

        aggregatePlayerToEra[playerId] = era

    cursor    .close()
    connection.close()

    aggregateEraList = []

    for key in aggregatePlayerToEra:
        aggregateEra = aggregatePlayerToEra[key]

        aggregateEraList.append(era)

        playerToAggregateEra[i+1][key] = aggregateEra
    
    aggregateEraList.sort()

    for key in aggregatePlayerToEra:
        aggregateEra = aggregatePlayerToEra[key]

        aggregateEraPercentile = get_reverse_percentile(weeklyEraList, era)

        playerToAggregatePercentile[i+1][key] = aggregateEraPercentile        

In [25]:
print(playerToAggregateEra[27]["luzaj001"])
print(playerToAggregatePercentile[27]["luzaj001"])

4.0754716981132075
81.70731707317073


In [8]:
#NEED ONE FOR STARTING PITCHERS AND ONE FOR RELIEVERS

eraQueryPartOne = "SELECT id,p_er,p_ipouts FROM pitching WHERE gametype = \"regular\" AND p_gs=\"1\" AND date BETWEEN \""
eraQueryPartTwo = "\" AND \""

addGradeStatement = "INSERT INTO weekly_grades (week_number,week_start,week_end,player_id,stat,num,percentile,season) VALUES("

playerToEraPercentile = {}
playerToEra           = {}

playerToQualityStartsPercentile = {}
playerToQualityStarts           = {}

#for i in range(len(weekStarts)):
for i in range(25,27):
    playerToEraPercentile[i+1] = {}
    playerToEra          [i+1] = {}

    playerToQualityStartsPercentile[i+1] = {}
    playerToQualityStarts          [i+1] = {}
    
    connection = sqlite3.connect('player_batting.db')
    cursor     = connection.cursor()

    weekStart = weekStarts[i]
    weekEnd   = weekEnds  [i]

    eraQuery = eraQueryPartOne + weekStart + eraQueryPartTwo + weekEnd + "\""

    cursor.execute(eraQuery)

    rows = cursor.fetchall()

    weeklyPlayerToEarnedRuns    = {}
    weeklyPlayerToOuts          = {}
    weeklyPlayerToQualityStarts = {}
    
    for row in rows:
        playerId    = row[0]
        earnedRuns  = float(row[1])
        outs        = float(row[2])

        if playerId in weeklyPlayerToEarnedRuns:
            curEarnedRuns = weeklyPlayerToEarnedRuns[playerId]
            weeklyPlayerToEarnedRuns[playerId] = curEarnedRuns + earnedRuns

            curOuts = weeklyPlayerToOuts[playerId]
            weeklyPlayerToOuts[playerId] = curOuts + outs
        else:
            weeklyPlayerToEarnedRuns[playerId] = earnedRuns
            weeklyPlayerToOuts      [playerId] = outs

        qualityStart = 0

        if earnedRuns <= 3.0 and outs >= 18.0:
            qualityStart = 1

        if playerId in weeklyPlayerToQualityStarts:
            weeklyPlayerToQualityStarts[playerId] = weeklyPlayerToQualityStarts[playerId] + qualityStart
        else:
            weeklyPlayerToQualityStarts[playerId] = qualityStart

    cursor    .close()
    connection.close()

    weeklyEraList          = []
    weeklyQualityStartList = []
    
    for key in weeklyPlayerToEarnedRuns:
        earnedRuns = weeklyPlayerToEarnedRuns[key]
        outs       = weeklyPlayerToOuts      [key]
        
        era = 0.0

        if outs == 0.0 and earnedRuns > 0.0:
            era = sys.float_info.max
        elif outs == 0.0 and earnedRuns == 0.0:
            era = 0.0
        else:
            era = (earnedRuns * 9.0) / (outs / 3.0)

        qualityStarts = weeklyPlayerToQualityStarts[key]
    
        weeklyQualityStartList.append(qualityStarts)
        weeklyEraList         .append(era)

    weeklyEraList         .sort()
    weeklyQualityStartList.sort()

    for key in weeklyPlayerToEarnedRuns:
        earnedRuns = weeklyPlayerToEarnedRuns[key]
        outs       = weeklyPlayerToOuts      [key]

        era = 0.0

        if outs == 0.0 and earnedRuns > 0.0:
            era = sys.float_info.max
            print(f"{i+1}, {key}")
            
        elif outs == 0.0 and earnedRuns == 0.0:
            era = 0.0
        else:
            era = (earnedRuns * 9.0) / (outs / 3.0)

        eraPercentile = get_reverse_percentile(weeklyEraList, era)

        qualityStarts = weeklyPlayerToQualityStarts[key]

        qualityStartsPercentile = get_percentile(weeklyQualityStartList, qualityStarts)

        playerToEra          [i+1][key] = era
        playerToEraPercentile[i+1][key] = eraPercentile

        playerToQualityStartsPercentile[i+1][key] = qualityStartsPercentile
        playerToQualityStarts          [i+1][key] = qualityStarts

        formattedAddEraGradeStatement = ""

        if era == sys.float_info.max:
            formattedAddEraGradeStatement = addGradeStatement + str(i+1) + "," + "\"" + weekStart + "\",\"" + weekEnd + "\",\"" + key + "\",\"STARTING_PITCHER_ERA\"," + "1e999" + "," + str(eraPercentile) + ", " + "2025" + ")"
        else:
            formattedAddEraGradeStatement = addGradeStatement + str(i+1) + "," + "\"" + weekStart + "\",\"" + weekEnd + "\",\"" + key + "\",\"STARTING_PITCHER_ERA\"," + str(era) + "," + str(eraPercentile) + ", " + "2025" + ")"
        
        formattedAddQualityStartsGradeStatement = addGradeStatement + str(i+1) + "," + "\"" + weekStart + "\",\"" + weekEnd + "\",\"" + key + "\",\"STARTING_PITCHER_QUALITY_STARTS\"," + str(qualityStarts) + "," + str(qualityStartsPercentile) + ", " + "2025" + ")"

        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()

        cursor.execute(formattedAddEraGradeStatement          )
        cursor.execute(formattedAddQualityStartsGradeStatement)
        
        connection.commit()
        
        cursor    .close()
        connection.close()        

In [ ]:
print(len(playerToQualityStarts[18]))
print(playerToQualityStartsPercentile[18]["gilbl002"])

In [ ]:
for key in playerToEraPercentile[17]:
    if playerToEraPercentile[17][key] > 80.0:
        print(f"{key}: {playerToEra[17][key]}, {playerToEraPercentile[17][key]}")

In [ ]:
for key in playerToQualityStartsPercentile[17]:
    if playerToQualityStartsPercentile[17][key] >= 66.0:
        print(f"{key}: {playerToQualityStarts[17][key]}, {playerToQualityStartsPercentile[17][key]}")

In [9]:
#NEED ONE FOR STARTING PITCHERS AND ONE FOR RELIEVERS

ksPerNineQueryPartOne = "SELECT id,p_k,p_ipouts FROM pitching WHERE gametype = \"regular\" AND p_gs=\"1\" AND date BETWEEN \""
ksPerNineQueryPartTwo = "\" AND \""

addGradeStatement = "INSERT INTO weekly_grades (week_number,week_start,week_end,player_id,stat,num,percentile,season) VALUES("

playerToKsPerNinePercentile = {}
playerToKsPerNine           = {}

#for i in range(len(weekStarts)):
for i in range(25,27):
    playerToKsPerNinePercentile[i+1] = {}
    playerToKsPerNine          [i+1] = {}
    
    connection = sqlite3.connect('player_batting.db')
    cursor     = connection.cursor()

    weekStart = weekStarts[i]
    weekEnd   = weekEnds  [i]

    ksPerNineQuery = ksPerNineQueryPartOne + weekStart + ksPerNineQueryPartTwo + weekEnd + "\""

    cursor.execute(ksPerNineQuery)

    rows = cursor.fetchall()

    weeklyPlayerToKs   = {}
    weeklyPlayerToOuts = {}
    
    for row in rows:
        playerId    = row[0]
        strikeOuts  = float(row[1])
        outs        = float(row[2])

        if playerId in weeklyPlayerToKs:
            curStrikeOuts = weeklyPlayerToKs[playerId]
            weeklyPlayerToKs[playerId] = curStrikeOuts + strikeOuts

            curOuts = weeklyPlayerToOuts[playerId]
            weeklyPlayerToOuts[playerId] = curOuts + outs
        else:
            weeklyPlayerToKs  [playerId] = strikeOuts
            weeklyPlayerToOuts[playerId] = outs

    cursor    .close()
    connection.close()

    weeklyKsPerNineList = []

    for key in weeklyPlayerToKs:
        strikeOuts = weeklyPlayerToKs  [key]
        outs       = weeklyPlayerToOuts[key]

        ksPerNine = 0.0

        if outs != 0.0:
            ksPerNine = (strikeOuts * 9.0) / (outs / 3.0)
        
        weeklyKsPerNineList.append(ksPerNine)

    weeklyKsPerNineList.sort()

    for key in weeklyPlayerToKs:
        strikeOuts = weeklyPlayerToKs  [key]
        outs       = weeklyPlayerToOuts[key]

        ksPerNine = 0.0

        if outs != 0.0:
            ksPerNine = (strikeOuts * 9.0) / (outs / 3.0)

        ksPerNinePercentile = get_percentile(weeklyKsPerNineList, ksPerNine)

        playerToKsPerNine          [i+1][key] = ksPerNine
        playerToKsPerNinePercentile[i+1][key] = ksPerNinePercentile

        formattedAddGradeStatement = addGradeStatement + str(i+1) + "," + "\"" + weekStart + "\",\"" + weekEnd + "\",\"" + key + "\",\"STARTING_PITCHER_STRIKE_OUTS_PER_NINE\"," + str(ksPerNine) + "," + str(ksPerNinePercentile) + ", " + "2025" + ")"

        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()

        cursor.execute(formattedAddGradeStatement)
        connection.commit()
        
        cursor    .close()
        connection.close()

In [ ]:
for key in playerToKsPerNinePercentile[17]:
    if playerToKsPerNinePercentile[17][key] > 80.0 and playerToPercentile[17][key] < 110.0:
        print(f"{key}: {playerToKsPerNine[17][key]}, {playerToKsPerNinePercentile[17][key]}")

In [ ]:
print(playerToKsPerNine[2]["gilbl002"])
print(playerToKsPerNinePercentile[2]["gilbl002"])

In [10]:
#NEED ONE FOR STARTING PITCHERS AND ONE FOR RELIEVERS

whipQueryPartOne = "SELECT id,p_w,p_h,p_ipouts FROM pitching WHERE gametype = \"regular\" AND p_gs=\"1\" AND date BETWEEN \""
whipQueryPartTwo = "\" AND \""

addGradeStatement = "INSERT INTO weekly_grades (week_number,week_start,week_end,player_id,stat,num,percentile,season) VALUES("

playerToWhipPercentile = {}
playerToWhip           = {}

#for i in range(len(weekStarts)):
for i in range(25,27):
    playerToWhipPercentile[i+1] = {}
    playerToWhip          [i+1] = {}

    playerToQualityStartsPercentile[i+1] = {}
    playerToQualityStarts          [i+1] = {}
    
    connection = sqlite3.connect('player_batting.db')
    cursor     = connection.cursor()

    weekStart = weekStarts[i]
    weekEnd   = weekEnds  [i]

    whipQuery = whipQueryPartOne + weekStart + whipQueryPartTwo + weekEnd + "\""

    cursor.execute(whipQuery)

    rows = cursor.fetchall()

    weeklyPlayerToWalks         = {}
    weeklyPlayerToHits          = {}
    weeklyPlayerToOuts          = {}
    weeklyPlayerToQualityStarts = {}
    
    for row in rows:
        playerId    = row[0]
        walks       = float(row[1])
        hits        = float(row[2])
        outs        = float(row[3])

        if playerId in weeklyPlayerToWalks:
            curWalks = weeklyPlayerToWalks[playerId]
            weeklyPlayerToWalks[playerId] = curWalks + walks

            curHits = weeklyPlayerToHits[playerId]
            weeklyPlayerToHits[playerId] = curHits + hits

            curOuts = weeklyPlayerToOuts[playerId]
            weeklyPlayerToOuts[playerId] = curOuts + outs
        else:
            weeklyPlayerToWalks[playerId] = walks
            weeklyPlayerToHits [playerId] = hits
            weeklyPlayerToOuts [playerId] = outs

    cursor    .close()
    connection.close()

    weeklyWhipList = []
    
    for key in weeklyPlayerToWalks:
        walks = weeklyPlayerToWalks[key]
        hits  = weeklyPlayerToHits [key]
        outs  = weeklyPlayerToOuts [key]
        
        whip = 0.0

        if outs == 0.0 and (walks > 0.0 or hits > 0.0):
            whip = sys.float_info.max
            print(f"{i+1}, {key}")
        elif outs == 0.0 and walks == 0.0 and hits == 0.0:
            whip = 0.0
        else:
            whip = (walks + hits) / (outs / 3.0)

        weeklyWhipList.append(whip)

    weeklyWhipList.sort()

    for key in weeklyPlayerToWalks:
        walks = weeklyPlayerToWalks[key]
        hits  = weeklyPlayerToHits [key]
        outs  = weeklyPlayerToOuts [key]
        
        whip = 0.0

        if outs == 0.0 and (walks > 0.0 or hits > 0.0):
            whip = sys.float_info.max
        elif outs == 0.0 and walks == 0.0 and hits == 0.0:
            whip = 0.0
        else:
            whip = (walks + hits) / (outs / 3.0)

        whipPercentile = get_reverse_percentile(weeklyWhipList, whip)

        playerToWhip          [i+1][key] = whip
        playerToWhipPercentile[i+1][key] = whipPercentile
        
        formattedAddGradeStatement = ""
        
        if whip == sys.float_info.max:
            formattedAddGradeStatement = addGradeStatement + str(i+1) + "," + "\"" + weekStart + "\",\"" + weekEnd + "\",\"" + key + "\",\"STARTING_PITCHER_WHIP\"," + "1e999" + "," + str(whipPercentile) + ", " + "2025" + ")"
        else:
            formattedAddGradeStatement = addGradeStatement + str(i+1) + "," + "\"" + weekStart + "\",\"" + weekEnd + "\",\"" + key + "\",\"STARTING_PITCHER_WHIP\"," + str(whip) + "," + str(whipPercentile) + ", " + "2025" + ")"

        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()

        cursor.execute(formattedAddGradeStatement)
        connection.commit()
        
        cursor    .close()
        connection.close()

In [ ]:
print(playerToWhip[2]["gilbl002"])
print(playerToWhipPercentile[2]["gilbl002"])

In [ ]:
for key in playerToWhipPercentile[17]:
    if playerToWhipPercentile[17][key] > 90.0:
        print(f"{key}: {playerToWhip[17][key]}, {playerToWhipPercentile[17][key]}")

In [ ]:
testQuery = "SELECT player_id,num FROM weekly_grades WHERE stat=\"STARTING_PITCHER_WHIP\" and num=1e999"

connection = sqlite3.connect('player_batting.db')
cursor     = connection.cursor()
cursor.execute(testQuery)
rows = cursor.fetchall()

for row in rows:
    playerId = row[0]
    num      = float(row[1])

    print(f"{playerId}, {num}")

cursor    .close()
connection.close()

In [11]:
eraStat           = "STARTING_PITCHER_ERA"
qualityStartsStat = "STARTING_PITCHER_QUALITY_STARTS"
ksPerNineStat     = "STARTING_PITCHER_STRIKE_OUTS_PER_NINE"
whipStat          = "STARTING_PITCHER_WHIP"

partialAggregateStatQuery = "SELECT stat,player_id,num FROM weekly_grades WHERE season=2025 AND (stat=\"STARTING_PITCHER_ERA\" OR stat=\"STARTING_PITCHER_QUALITY_STARTS\" OR stat=\"STARTING_PITCHER_STRIKE_OUTS_PER_NINE\" OR stat=\"STARTING_PITCHER_WHIP\") AND week_number="
addGradeStatement         = "INSERT INTO weekly_grades (week_number,week_start,week_end,player_id,stat,num,percentile,season) VALUES("

weeklyGrades     = {}
weeklyPercentile = {}

#for i in range(1,18):
for i in range(26,28):
    weeklyGrades    [i] = {}
    weeklyPercentile[i] = {}
    
    aggregateStatQuery = partialAggregateStatQuery + str(i)

    connection = sqlite3.connect('player_batting.db')
    cursor     = connection.cursor()

    cursor.execute(aggregateStatQuery)

    rows = cursor.fetchall()

    maxEra = 0.0
    minEra = sys.float_info.max

    maxQualityStarts = 0.0
    minQualityStarts = 10000.0

    maxKsPerNine = 0.0
    minKsPerNine = 10000.0

    maxWhip = 0.0
    minWhip = sys.float_info.max

    weekToOverallEra           = {}
    weekToOverallQualityStarts = {}
    weekToOverallKsPerNine     = {}
    weekToOverallWhip          = {}

    weeklyPlayerIds = set()
    
    for row in rows:
        stat     = row[0]
        playerId = row[1]
        num      = float(row[2])

        if num == float('inf'):
            num = sys.float_info.max

        weeklyPlayerIds.add(playerId)
        
        if stat == eraStat:
            weekToOverallEra[playerId] = num
            maxEra = max(num, maxEra)
            minEra = min(num, minEra)
        elif stat == qualityStartsStat:
            weekToOverallQualityStarts[playerId] = num
            maxQualityStarts = max(num, maxQualityStarts)
            minQualityStarts = min(num, minQualityStarts)
        elif stat == ksPerNineStat:
            weekToOverallKsPerNine[playerId] = num
            maxKsPerNine = max(num, maxKsPerNine)
            minKsPerNine = min(num, minKsPerNine)
        elif stat == whipStat:
            weekToOverallWhip[playerId] = num
            maxWhip = max(num, maxWhip)
            minWhip = min(num, minWhip)

    cursor    .close()
    connection.close()

    overallGrades = []

    print(f"{i}, {maxEra}, {minEra}, {maxQualityStarts}, {minQualityStarts}, {maxKsPerNine}, {minKsPerNine}, {maxWhip}, {minWhip}")
    
    for playerId in weeklyPlayerIds:
        era           = 0.0
        qualityStarts = 0
        ksPerNine     = 0.0
        whip          = 0.0

        if playerId in weekToOverallEra:
            era = weekToOverallEra[playerId]
        if playerId in weekToOverallQualityStarts:
            qualityStarts = weekToOverallQualityStarts[playerId]
        if playerId in weekToOverallKsPerNine:
            ksPerNine = weekToOverallKsPerNine[playerId]
        if playerId in weekToOverallWhip:
            whip = weekToOverallWhip[playerId]

        normalizedEra           = reverse_normalization(era,             maxEra,           minEra          )
        normalizedQualityStarts = normalize            (qualityStarts,   maxQualityStarts, minQualityStarts)
        normalizedKsPerNine     = normalize            (ksPerNine,       maxKsPerNine,     minKsPerNine    )
        normalizedWhip          = reverse_normalization(whip,            maxWhip,          minWhip         )

        overallGrade = normalizedEra + normalizedQualityStarts + normalizedKsPerNine + normalizedWhip

        weeklyGrades[i][playerId] = overallGrade

        overallGrades.append(overallGrade)
    
    overallGrades.sort()

    overallGradeWeekStart = weekStarts[i-1]
    overallGradeWeekEnd   = weekEnds  [i-1]
    
    for playerId in weeklyPlayerIds:
        grade = weeklyGrades[i][playerId]

        percentile = get_percentile(overallGrades, grade)

        weeklyPercentile[i][playerId] = percentile

        formattedAddGradeStatement = addGradeStatement + str(i) + "," + "\"" + overallGradeWeekStart + "\",\"" + overallGradeWeekEnd + "\",\"" + playerId + "\",\"STARTING_PITCHER_TOTAL\"," + str(grade) + "," + str(percentile) + ", " + "2025" + ")"

        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()

        cursor.execute(formattedAddGradeStatement)
        connection.commit()
        
        cursor    .close()
        connection.close() 

26, 81.0, 0.0, 2.0, 0.0, 19.8, 0.0, 10.5, 0.2222222222222222
27, 94.5, 0.0, 2.0, 0.0, 27.0, 0.0, 12.0, 0.0


In [ ]:
weekStarts[18-1]